In [ ]:
# Imports
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "normal"

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# srun -w ruapehu -c 20 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha
# ssh -L 8889:localhost:8890 ls985@ruapehu

In [ ]:
vector = [644.3815324240131, 635.8354316360783, 652.6871707260143]
vector = [v / 60 for v in vector]
print(np.mean(vector), np.std(vector))

In [ ]:
vector = [523.6875479070004, 523.1290393669624, 527.9661321379244]
vector = [v / 60 for v in vector]
print(np.mean(vector), np.std(vector))

In [ ]:
vector = [1746.206744691939, 1748.5203055860475, 1741.463884711964]
vector = [v / 60 for v in vector]
print(np.mean(vector), np.std(vector))

In [ ]:
vector = [2722.0670927340398, 2736.0910655630287, 2736.1201942099724]
vector = [v / 60 for v in vector]
print(np.mean(vector), np.std(vector))

In [ ]:
vector = [1515.9566779749002, 1549.8519769191043, 1527.2650793070206]
vector = [v / 60 for v in vector]
print(np.mean(vector), np.std(vector))

In [ ]:
vector = [6475.597678093938, 6498.888374426984, 6527.978125871043]
vector = [v / 60 for v in vector]
print(np.mean(vector), np.std(vector))

In [ ]:
# Create experiment time data
experiment_time = [
    [
        152.661791191902,
        96.17854976502713,
        77.31260278297123,
        68.22936446906533,
        63.953428369015455,
        65.17627378297038,
        67.29301187500823,
        67.23576708394103,
        68.60667291597929,
        68.85605030797888,
        73.7188119860366,
        75.6840253919363,
        77.40147036302369,
        79.82236306695268,
        83.1808044660138,
        84.78081731800921,
    ],
    [
        163.5559520300012,
        103.37907770101447,
        78.47813307400793,
        82.58496832894161,
        66.51790506893303,
        67.24684307095595,
        75.8940713009797,
        70.0047331439564,
        74.5334963009227,
        72.590588002,
        78.36495467904024,
        78.1230543279089,
        77.96214476507157,
        91.62520050699823,
        94.7607045309851,
        94.71368389390409,
    ],
    [
        151.56941861705855,
        96.06653415004257,
        76.406124445959,
        68.08188454597257,
        65.19329527905211,
        65.24129687401,
        65.11055661505088,
        66.32311544299591,
        65.4763042019913,
        69.149786180933,
        73.20346314599738,
        74.40108335099649,
        80.23279881500639,
        78.42827577597927,
        82.72695306804962,
        84.0946367509896,
    ],
]
# Create a dictionary with your data
list_of_dataframes: list[pd.DataFrame] = []
for exp in experiment_time:
    data = {"n_workers": range(16), "experiment_time": exp}

    # Create a pandas DataFrame from the dictionary
    list_of_dataframes.append(pd.DataFrame(data))
experiment_time_dataframe = pd.concat(list_of_dataframes)

In [ ]:
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
dataframes: list[pd.DataFrame] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob(
    "clients_training_stats_conc-est-cifar-1gpu-a40-*.csv"
):
    # Read the .csv file into a pandas DataFrame
    clients_training_stats_df = pd.read_csv(file)
    # Remove outliers (delta >= 0.1)
    clients_training_stats_df = clients_training_stats_df[
        clients_training_stats_df["delta"] < 0.1  # noqa: PLR2004
    ]
    # Extract the number of workers from the filename (format `*-n.csv`)
    n = int(file.stem.split("-")[-1].split("_")[-1].replace(".csv", ""))
    # Add a new column for 'n'
    clients_training_stats_df["n"] = n
    # Append the DataFrame to the list
    dataframes.append(clients_training_stats_df)
# Concatenate all the DataFrames in the list
complete_dataframe = pd.concat(dataframes)
colors = sns.color_palette("tab10", n_colors=2)
current_color = colors.pop(0)
# Filter the dataframe to keep only `n` column with power of two values
complete_dataframe = complete_dataframe[complete_dataframe["n"] <= 16]  # noqa: PLR2004
# Create the figure
plt.figure(figsize=(6, 5))
# Create a boxplot with 'n' on the x-axis and 'delta' on the y-axis
patches = []
ax1 = sns.boxplot(
    x="n",
    y="delta",
    data=complete_dataframe,
    showfliers=False,
    boxprops={"facecolor": current_color, "alpha": 0.8},
    flierprops={
        "markerfacecolor": current_color,
        "marker": "o",
        "markersize": 5,
        "linestyle": "none",
        "alpha": 0.8,
    },
)
patches.append(mpatches.Patch(color=current_color, label="Client Training Time [s]"))
# Add axis labels
x_label = plt.xlabel("Number of Workers")
x_label.set_weight("bold")
y_label = plt.ylabel("Client Training Time [s]")
y_label.set_weight("bold")
# Set the y-ticks for the second axis
y_min, y_max = ax1.get_ylim()
ax1.set_yticks(
    np.linspace(y_min, y_max, 6),
    labels=[f"{y:.2f}" for y in np.linspace(y_min, y_max, 6)],
)
plt.twinx()
# Extract the last color
current_color = colors.pop(0)
# Plot `experiment_time` vs `n_workers` as a scatterplot in the current panel.
# plt.plot(n_workers, experiment_time, marker="o", color=current_color)
sns.lineplot(
    data=experiment_time_dataframe,
    y="experiment_time",
    x="n_workers",
    color=current_color,
    err_style="band",
    marker="X",
)
patches.append(mpatches.Patch(color=current_color, label="Experiment Time [s]"))
# Add label for the right x axis
y_label = plt.ylabel("Experiment Time [s]")
y_label.set_weight("bold")
# Add the legend manually
legend = plt.legend(loc="upper center", bbox_to_anchor=(0.4, 1.0), handles=patches)
plt.setp(legend.get_texts(), fontweight="bold")
plt.grid(axis="y")
# Set the y-ticks for the second axis
ax2 = plt.gca()
y_min, y_max = ax2.get_ylim()
ax2.set_yticks(
    np.linspace(y_min, y_max, 6),
    labels=[f"{int(y / 10) * 10}" for y in np.linspace(y_min, y_max, 6)],
)
# Save the plot
plt.savefig("concurrency_optimum.pdf", format="pdf", dpi=800, bbox_inches="tight")
# Show the plot
plt.show()

In [ ]:
def analyse_gpu_metrics(csv_file: Path) -> pd.DataFrame:
    """Analyse the GPU metrics from a .csv file and return a pandas DataFrame."""
    # Read the .csv file into a pandas DataFrame
    gpu_dataframe = pd.read_csv(
        csv_file,
        header=None,
        names=["index", "gpu_util", "mem_total", "mem_used", "mem_free", "timestamp"],
        skipfooter=1,
        engine="python",
    )
    # Create a new column for the memory used fraction
    gpu_dataframe["mem_used_frac"] = (
        gpu_dataframe["mem_used"] / gpu_dataframe["mem_total"]
    )
    # Drop the columns that are not needed
    gpu_dataframe = gpu_dataframe.drop(columns=["index", "mem_free", "mem_total"])
    # Add column for the number of workers reading from the filename
    n = int(Path(csv_file).stem.split("-")[-1].split("_")[-1].replace(".csv", ""))
    gpu_dataframe["n"] = n
    # Convert the `timestamp` column to datetime
    gpu_dataframe["timestamp"] = pd.to_datetime(gpu_dataframe["timestamp"])
    # Convert `timestamp` to seconds
    gpu_dataframe["timestamp"] = gpu_dataframe["timestamp"].astype(int) / 10**9
    # Filter-out rows where `mem_used` is lower than 100 (MB)
    gpu_dataframe = gpu_dataframe[gpu_dataframe["mem_used"] > 100]  # noqa: PLR2004
    gpu_dataframe = gpu_dataframe[
        gpu_dataframe["mem_used"] > gpu_dataframe["mem_used"].min()
    ]
    # Shift the timestamp to start from 0
    gpu_dataframe["timestamp"] = (
        gpu_dataframe["timestamp"] - gpu_dataframe["timestamp"].min()
    )
    # Compute AUC for `gpu_util` and `mem_used` columns over `timestamp`
    gpu_dataframe["gpu_util_auc"] = (
        np.trapz(gpu_dataframe["gpu_util"], gpu_dataframe["timestamp"])
        / gpu_dataframe["timestamp"].max()
    )
    gpu_dataframe["mem_used_frac_auc"] = (
        np.trapz(gpu_dataframe["mem_used_frac"], gpu_dataframe["timestamp"])
        / gpu_dataframe["timestamp"].max()
    )
    return gpu_dataframe

In [ ]:
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
dataframes: list[pd.DataFrame] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("gpu-*.csv"):
    # Read the .csv file into a pandas DataFrame
    gpu_dataframe = analyse_gpu_metrics(file)
    # Append the DataFrame to the list
    dataframes.append(gpu_dataframe)
# Concatenate all the DataFrames in the list
complete_dataframe = pd.concat(dataframes)
# Remove row with n_workers > 16
complete_dataframe = complete_dataframe[complete_dataframe["n"] <= 16]  # noqa: PLR2004
# Sort by 'n'
complete_dataframe = complete_dataframe.sort_values(by="n")
# Create the patches list
patches = []
# Create the color palette
colors = sns.color_palette("tab10", n_colors=2)
# Extract the first color
current_color = colors.pop(0)
# Create a scatter plot with 'n' on the x-axis and 'gpu_util_auc' on the y-axis
plt.figure(figsize=(6, 5))
plt.plot(
    complete_dataframe["n"].astype(int),
    complete_dataframe["gpu_util_auc"],
    marker="o",
    label="Avg. GPU utilization [%]",
    color=current_color,
)
patches.append(mpatches.Patch(color=current_color, label="Avg. GPU utilization [%]"))
# Extract the axis object
ax1 = plt.gca()
# Set the x-ticks
ax1.set_xticks(
    np.linspace(1, 16, 16),
    labels=[f"{int(y)}" for y in np.linspace(1, 16, 16)],
)
# Set the y-ticks
y_min, y_max = ax1.get_ylim()
ax1.set_yticks(
    np.linspace(y_min, y_max, 6),
    labels=[f"{int(y / 10) * 10}" for y in np.linspace(y_min, y_max, 6)],
)
plt.grid(axis="both", linestyle="--")
x_label = plt.xlabel("Number of Workers")
x_label.set_weight("bold")
y_label = plt.ylabel("Avg. GPU utilization [%]")
y_label.set_weight("bold")
plt.twinx()
# Extract the first color
current_color = colors.pop(0)
plt.plot(
    complete_dataframe["n"].astype(int),
    complete_dataframe["mem_used_frac_auc"] * 100,
    marker="o",
    label="Allocated GPU Memory [%]",
    color=current_color,
)
patches.append(mpatches.Patch(color=current_color, label="Allocated GPU Memory [%]"))
# Extract the axis object
ax2 = plt.gca()
# Set the y-ticks
y_min, y_max = ax2.get_ylim()
ax2.set_yticks(
    np.linspace(y_min, y_max, 6),
    labels=[f"{int(y)}" for y in np.linspace(y_min, y_max, 6)],
)
plt.grid(axis="both", linestyle="--")
y_label = plt.ylabel("Allocated GPU Memory [%]")
y_label.set_weight("bold")
# Save the plot
plt.savefig("concurrency_gpu_metrics.pdf", format="pdf", dpi=800, bbox_inches="tight")
# Add the legend manually
legend = plt.legend(loc="lower right", handles=patches)
plt.setp(legend.get_texts(), fontweight="bold")
# Show the plot
plt.show()

In [ ]:
"""
Write a function that separately plots the second and the thirds column versus the first column from records in a .csv file formatted like:
    Timestamp,Memory,CPU,kB_rd/s,kB_wr/s,kB_ccwr/s,I/O Delay,USR_MS,SYTEM_MS,GUEST_MS
    1713450516520,0,0,0,0,0,0,130,100,0
    1713450516762,0,0,0,0,0,0,270,120,0
    1713450516993,0,0,0,0,0,0,450,130,0
"""


def analise_system_metrics(csv_file: Path) -> pd.DataFrame:
    # Read the .csv file into a pandas DataFrame
    system_metrics_dataframe = pd.read_csv(csv_file)
    # Get the n workers from the filename
    n = int(Path(csv_file).stem.split("-")[-1].split("_")[-1].replace(".csv", ""))
    system_metrics_dataframe["n"] = n
    # Shift the timestamp to start from 0
    system_metrics_dataframe["Timestamp"] = (
        system_metrics_dataframe["Timestamp"]
        - system_metrics_dataframe["Timestamp"].min()
    )
    # Convert the timestamp from ms to s
    system_metrics_dataframe["Timestamp"] = system_metrics_dataframe["Timestamp"] / 1000
    # Remove the last 5 rows
    # system_metrics_dataframe = system_metrics_dataframe[:-int(3 * n)]
    return system_metrics_dataframe

In [ ]:
ROOT_PATH = "/nfs-share/ls985/projects/sosp-pollen_worker/"
dataframes: list[pd.DataFrame] = []
# Iterate over all the .csv files in the directory
for file in Path(ROOT_PATH).glob("pidstat-ttri*.csv"):
    dataframes.append(analise_system_metrics(file))
# Concatenate all the DataFrames in the list
complete_dataframe = pd.concat(dataframes)
complete_dataframe = complete_dataframe.reindex()
# Sort by 'n'
complete_dataframe = complete_dataframe.sort_values(by="n")
# Plot the second and third columns versus the first column
for col in ["Memory", "CPU", "kB_rd/s", "kB_wr/s", "kB_ccwr/s"]:
    # plt.figure(figsize=(10, 6))
    # plt.plot(complete_dataframe['Timestamp'], complete_dataframe[col], label)
    for nn in complete_dataframe["n"].unique():
        # Get the dataframe woth the current number of workers
        tmp_dataframe = complete_dataframe[complete_dataframe["n"] == nn]
        # Sort by timestamp
        tmp_dataframe = tmp_dataframe.sort_values(by="Timestamp")
        plt.plot(tmp_dataframe["Timestamp"], tmp_dataframe[col], label=f"n={nn}")
    plt.xlabel("Time [s]")
    plt.ylabel(col)
    plt.grid()
    plt.legend()
    plt.show()
for col in ["I/O Delay", "USR_MS", "SYTEM_MS", "GUEST_MS"]:
    # plt.figure(figsize=(10, 6))
    # plt.plot(complete_dataframe['Timestamp'], complete_dataframe[col], label)
    # Aggregate sum the column col grouping by n
    tmp_dataframe = complete_dataframe.groupby("n")[col].max()
    # Draw lineplot with the aggregated data
    plt.plot(tmp_dataframe.index, tmp_dataframe.values, marker="o")
    print(tmp_dataframe.head())

    # plt.boxplot(
    # for nn in complete_dataframe['n'].unique():
    #     # Get the dataframe woth the current number of workers
    #     tmp_dataframe = complete_dataframe[complete_dataframe['n'] == nn]
    #     # Sort by timestamp
    #     tmp_dataframe = tmp_dataframe.sort_values(by="Timestamp")
    #     plt.boxplot(tmp_dataframe["Timestamp"], tmp_dataframe[col], label=f"n={nn}")
    # plt.xlabel("Time [s]")
    # plt.ylabel(col)
    # plt.grid()
    # plt.legend()
    plt.show()